In [1]:
import torch
from torch import nn

In [2]:
x = torch.randn(2, 5, 64)

In [3]:
B,T,d_model = x.shape

In [4]:
def split_heads(x,num_heads):
  B,T,d_model = x.shape
  d_head = d_model // num_heads
  x = (x.view(B,T,num_heads,-1)).transpose(1,2)
  return x.contiguous()

In [5]:
def merge_heads(x):
  B,num_heads,T,d_head = x.shape
  x = x.transpose(1,2).reshape(B,T,-1)
  return x

In [6]:
print(split_heads(x,4).shape)

torch.Size([2, 4, 5, 16])


In [7]:
heads = split_heads(x,4)
print(merge_heads(heads).shape)

torch.Size([2, 5, 64])


In [8]:
def masked_attention(q, k, v, mask=None):
    dk = (q.size(-1)) ** 0.5
    scores = (q @ k.transpose(-2, -1)) / dk
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights

In [9]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads

    self.q_proj = nn.Linear(d_model,d_model)
    self.k_proj = nn.Linear(d_model,d_model)
    self.v_proj = nn.Linear(d_model,d_model)
    self.out_proj = nn.Linear(d_model,d_model)

  def forward(self,x,mask=None):
    q = self.q_proj(x)
    k = self.k_proj(x)
    v = self.v_proj(x)

    q = split_heads(q,self.num_heads)
    k = split_heads(k,self.num_heads)
    v = split_heads(v,self.num_heads)

    if mask is not None:
      if mask.dim() == 3:
        mask = mask.unsqueeze(1)
    attn_output, attn_weights = masked_attention(q, k, v, mask)
    attn_output = merge_heads(attn_output)
    return self.out_proj(attn_output)

In [11]:
class SimpleTransformationBlock(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.attn = MultiHeadAttention(d_model,num_heads)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(
        nn.Linear(d_model,d_model),
        nn.ReLU(),
        nn.Linear(d_model,d_model)
    )
  def forward(self,x,mask=None):
    out1 = self.norm1(x)
    attn_out = self.attn(out1,mask)
    h = x + attn_out
    out2 = self.norm2(h)
    ffn_out = self.ffn(out2)
    return h + ffn_out

In [12]:
def make_causal_mask(seq_len):
    # 生成 [T, T] 的下三角布尔矩阵
    return torch.tril(torch.ones(seq_len, seq_len)).bool()

In [20]:
class TinyDecoder(nn.Module):
  def __init__(self, vocab_size, d_model=32, num_heads=4, max_len=32):
    super().__init__()
    self.token_embedding = nn.Embedding(vocab_size,d_model)
    self.pos_embedding = nn.Embedding(max_len,d_model)
    self.block = SimpleTransformationBlock(d_model,num_heads)
    self.lm_head = nn.Linear(d_model,vocab_size)

  def forward(self, token_ids):
    T = token_ids.shape[1]
    index = torch.arange(T,device=token_ids.device)
    x = self.token_embedding(token_ids)
    pos = self.pos_embedding(index)
    x = x + pos
    mask = make_causal_mask(T)
    x = self.block(x,mask)
    logits = self.lm_head(x)
    return logits

In [23]:
@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens=5):
  for _ in range(max_new_tokens):
    logits = model(prompt_ids)
    pos = logits[:,-1,:]
    nextid = torch.argmax(pos,dim=-1)
    posnext = nextid.unsqueeze(-1)
    prompt_ids = torch.cat((prompt_ids,posnext),dim=-1)
  return prompt_ids

In [24]:
prompt = torch.tensor([[1, 2, 3]])
model = TinyDecoder(vocab_size=100)
result = generate(model, prompt, max_new_tokens=5)
print(result.shape) # 预期结果: [1, 8]

torch.Size([1, 8])
